<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/deberta_performances_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using L4 GPU

In [ ]:
from google.colab import drive
from google.colab import auth
import os
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Authenticate Google User
auth.authenticate_user()

# 3. Install necessary libraries
# 'sentencepiece' is required for DeBERTa V3 tokenizer
!pip install -q accelerate transformers datasets scikit-learn matplotlib sentencepiece protobuf

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import pandas as pd
from torch.utils.data import Dataset

class TestDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Convert item to tensor on the fly
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def read_csv_file(file_path):
    try:
        # Read columns ensuring string type for body
        data = pd.read_csv(file_path, names=['body', 'inflation'], header=0, dtype={'body': str, 'inflation': str})
        data = data.dropna(subset=['inflation'])
        return data
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Path to test data
file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv'

# Read data
test_data = read_csv_file(file_path)

if test_data is not None:
    # No prompt formatting for DeBERTa - using raw body text
    print(f"Data loaded successfully. Number of samples: {len(test_data)}")

    # Verify the first sample (raw text)
    print("\nSample input (Raw Text):")
    print(test_data['body'].iloc[0][:200] + "...")
else:
    raise ValueError("Failed to load test data.")

# 1,039

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# DIRECTORY PATH: Update 'checkpoint-XXX' to your actual best checkpoint folder
# Example: .../deberta-large-fine-tuning-1040/checkpoint-312/
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-1039-rerun/checkpoint-490/'

# If the specific checkpoint folder doesn't exist, try the root output dir (if trainer saved final there)
if not os.path.exists(checkpoint_path):
    print(f"Warning: Specific checkpoint path not found: {checkpoint_path}")
    print("Attempting to load from base directory...")
    checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-1039-rerun/'

print(f"Loading tokenizer and model from: {checkpoint_path}")

try:
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, local_files_only=True)
except OSError as e:
    print(f"Error loading tokenizer: {e}")
    # Fallback to downloading base tokenizer if local artifacts are missing
    print("Fallback: Loading base tokenizer config...")
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        num_labels=3,
        # DeBERTa works well with standard float32 or float16/bfloat16
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
        device_map="auto",
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")



In [ ]:
from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['body'].tolist(), # Using raw body
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=16) # DeBERTa is lighter than Llama, can handle larger batch

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 16} / {len(test_dataset)} samples...")

print("Inference completed.")

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

# 520

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# DIRECTORY PATH
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-520-rerun/checkpoint-343/'

# ☆
# Checkpoint check logic...
if not os.path.exists(checkpoint_path):
    print(f"Warning: Specific checkpoint path not found: {checkpoint_path}")
    print("Attempting to load from base directory...")
    checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-520-rerun/'

print(f"Loading model from: {checkpoint_path}")

print("Loading tokenizer from microsoft/deberta-v3-large...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        num_labels=3,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
        device_map="auto",
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['body'].tolist(), # Using raw body
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=16) # DeBERTa is lighter than Llama, can handle larger batch

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 16} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

# 260

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# DIRECTORY PATH
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-260-rerun/checkpoint-175/'

# Checkpoint check logic...
# ☆
if not os.path.exists(checkpoint_path):
    print(f"Warning: Specific checkpoint path not found: {checkpoint_path}")
    print("Attempting to load from base directory...")
    checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-260-rerun/'

print(f"Loading model from: {checkpoint_path}")

print("Loading tokenizer from microsoft/deberta-v3-large...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        num_labels=3,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
        device_map="auto",
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['body'].tolist(), # Using raw body
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=16) # DeBERTa is lighter than Llama, can handle larger batch

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 16} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

#130

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# DIRECTORY PATH
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-130-rerun/checkpoint-150/'

# ☆
# Checkpoint check logic...
if not os.path.exists(checkpoint_path):
    print(f"Warning: Specific checkpoint path not found: {checkpoint_path}")
    print("Attempting to load from base directory...")
    checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-130-rerun/'

print(f"Loading model from: {checkpoint_path}")

print("Loading tokenizer from microsoft/deberta-v3-large...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        num_labels=3,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
        device_map="auto",
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['body'].tolist(), # Using raw body
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=16) # DeBERTa is lighter than Llama, can handle larger batch

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 16} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")

# 65

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# DIRECTORY PATH
# ☆
checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-65-rerun/checkpoint-100/'

# ☆
# Checkpoint check logic...
if not os.path.exists(checkpoint_path):
    print(f"Warning: Specific checkpoint path not found: {checkpoint_path}")
    print("Attempting to load from base directory...")
    checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/deberta-large-fine-tuning-65-rerun/'

print(f"Loading model from: {checkpoint_path}")

print("Loading tokenizer from microsoft/deberta-v3-large...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        num_labels=3,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
        device_map="auto",
        local_files_only=True
    )
except OSError as e:
    print(f"Error loading model: {e}")
    raise

model.eval()
print("Model loaded successfully.")

from torch.utils.data import DataLoader

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['body'].tolist(), # Using raw body
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    print("Warning: Found non-numeric labels. attempting to coerce...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=16) # DeBERTa is lighter than Llama, can handle larger batch

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        # Move inputs to device
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        # Forward pass
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        # Store results
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 16} / {len(test_dataset)} samples...")

print("Inference completed.")

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Macro Average
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
# Micro Average (Same as Accuracy for single-label)
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")